# 技能1 · Day 2 上机：营销数据表示实战 + 多模态大模型演进

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **sentence-transformers** 将客户行为文本/产品描述/营销文案编码为语义向量，做检索和分群
2. 用 **PyTorch** 实现 Two-Tower 双塔模型，理解 InfoNCE 对比损失让客户-产品向量对齐
3. 用 **transformers CLIPModel** 做图文对齐（产品图-文匹配），理解对比学习双塔架构
4. 梳理 CLIP -> BLIP-2 -> GPT-4o -> LLaVA 的多模态演进路线，指出各阶段营销应用

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：sentence-transformers（文本embedding）/ transformers CLIP（图文对齐）/ torch（Two-Tower）。
营销映射：为美妆电商构建客户/产品/内容/跨域四大表示。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ 首次运行会自动下载预训练模型（sentence-transformers 约470MB，CLIP 约600MB）。
> CLIP 模型较大，CPU推理约2-5秒/张图片。如遇下载慢可设 `HF_ENDPOINT=https://hf-mirror.com`。

In [ ]:
# !pip install sentence-transformers transformers torch scikit-learn pillow -q
# export HF_ENDPOINT=https://hf-mirror.com  # 国内镜像加速

## 1. 数据集背景与营销映射

**场景**：美妆电商，需要为四类核心对象构建向量化表示。

| 数据类型 | 内容 | 数量 | TODO |
|---------|------|:----:|------|
| 客户行为文本 | 浏览/搜索/购买行为描述 | 8条 | TODO1: 客户embedding+KMeans分群 |
| 产品描述 | 美妆产品标题+功效+成分+价格 | 8个 | TODO2: 产品embedding+cosine检索 |
| 营销文案 | 小红书种草风格文案 | 6条 | TODO3: 内容embedding+相似推荐 |
| 客户-产品交互 | 正样本购买对 | 8对 | TODO4: Two-Tower训练 |
| 产品图片 | PIL合成色块图片 | 3张 | TODO5: CLIP图文对齐 |

**四大表示类型**：
- **客户嵌入**：行为序列/交易/文本反馈 -> 统一向量 -> 分群/推荐
- **产品嵌入**：属性/描述/图片 -> 向量 -> 相似检索/交叉推荐
- **内容嵌入**：广告文案/图文 -> 向量 -> 相似推荐/内容检索
- **跨域对齐**：客户-产品跨空间 -> Two-Tower对比学习 -> 匹配检索

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor

# ============================================================
# 美妆电商营销数据（内嵌，无需外部下载）
# ============================================================
customers = [
    {"id": "C001", "text": "搜索了提亮精华 浏览了烟酰胺亮肤精华液 查看烟酰胺浓度 比较了3个品牌 加入收藏 7天后购买"},
    {"id": "C002", "text": "搜索了哑光口红 浏览了丝绒哑光口红 查看色号 试色视频 直接购买 购买了2支"},
    {"id": "C003", "text": "搜索了防晒霜 浏览了清透防晒霜 查看SPF指数 加入购物车 3天后收到优惠券然后购买"},
    {"id": "C004", "text": "搜索了保湿面霜 浏览了玻尿酸保湿面霜 查看成分 比较了兰蔻和雅诗兰黛 购买 写了好评"},
    {"id": "C005", "text": "搜索了清洁面膜 浏览了水杨酸清洁面膜 查看控油效果 直接购买 推荐给朋友"},
    {"id": "C006", "text": "搜索了抗老眼霜 浏览了视黄醇抗老眼霜 查看视黄醇浓度 加入收藏 5天后购买 购买后晒单"},
    {"id": "C007", "text": "搜索了美白精华 浏览了维C亮肤安瓶 查看维C浓度 比较了3个品牌 购买 写了好评"},
    {"id": "C008", "text": "搜索了敏感肌乳液 浏览了神经酰胺修护乳液 查看成分 加入购物车 2天后购买"},
]

products = [
    {"id": "P001", "name": "烟酰胺亮肤精华液", "desc": "5%烟酰胺浓度 提亮肤色 收缩毛孔 适合油性肌肤 28天见效 售价199元"},
    {"id": "P002", "name": "丝绒哑光口红", "desc": "哑光丝绒质地 持久不脱色 8色可选 一抹显白 售价198元"},
    {"id": "P003", "name": "清透防晒霜", "desc": "SPF50+ PA++++ 防晒黑 质地轻薄不闷痘 50ml 售价159元"},
    {"id": "P004", "name": "玻尿酸保湿面霜", "desc": "2%玻尿酸 深层保湿 适合干性肌肤 温和不刺激 售价259元"},
    {"id": "P005", "name": "水杨酸清洁面膜", "desc": "2%水杨酸 深层清洁 控油 每周使用两次 售价129元"},
    {"id": "P006", "name": "视黄醇抗老眼霜", "desc": "0.3%视黄醇 淡化细纹 紧致眼周 夜间使用 售价399元"},
    {"id": "P007", "name": "维C亮肤安瓶", "desc": "10%维C 抗氧化 提亮肤色 晨间使用 售价299元"},
    {"id": "P008", "name": "神经酰胺修护乳液", "desc": "神经酰胺 修护皮肤屏障 敏感肌可用 温和配方 售价179元"},
]

marketing_copy = [
    "姐妹们！这款烟酰胺精华液真的绝了 5%浓度提亮肤色 28天见效 油皮也能冲",
    "哑光丝绒口红 一抹显白 持久不脱色 8色可选 聚会必囤",
    "夏天必备 SPF50+防晒霜 轻薄不闷痘 海边玩水也不怕 快冲",
    "干皮亲妈 玻尿酸保湿面霜 深层补水 第二天脸蛋软软的",
    "大油田福音 水杨酸清洁面膜 每周两次 毛孔干干净净",
    "抗老早开始 视黄醇眼霜 淡化细纹 28天紧致眼周",
]

print(f"数据准备完成: {len(customers)}个客户, {len(products)}个产品, {len(marketing_copy)}条文案")

## 1：客户embedding（行为文本->向量+KMeans分群）

In [ ]:
# 1. 客户embedding（行为文本->向量+KMeans分群+Silhouette选K）
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
customer_texts = [c["text"] for c in customers]
customer_embeddings = model.encode(customer_texts, show_progress_bar=False)

# 用Silhouette Score自动选择最优K
best_k, best_score = 2, -1
for k in range(2, min(5, len(customers))):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(customer_embeddings)
    score = silhouette_score(customer_embeddings, labels)
    print(f"  K={k}, Silhouette={score:.4f}")
    if score > best_score:
        best_k, best_score = k, score

# 用最优K做最终分群
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
customer_clusters = kmeans.fit_predict(customer_embeddings)

print(f"\n客户embedding维度: {customer_embeddings.shape}")
print(f"最优K值: {best_k}, Silhouette: {best_score:.4f}")
for c, cluster in zip(customers, customer_clusters):
    print(f"  {c['id']}: 群组{cluster} | {c['text'][:30]}...")

## 2. 产品表示与内容表示

客户 embedding 解决"客户是谁"的问题。接下来：
- **产品 embedding**：把产品描述编码为向量，支持"以文搜产品"（输入需求文本，检索最匹配的产品）
- **内容 embedding**：把营销文案编码为向量，支持"相似文案推荐"（给定一条文案，找风格最相似的其他文案）

两者都用 sentence-transformers 编码 + cosine 相似度检索，是表示工程最基础的应用。

## 2-3：产品embedding + 内容embedding

In [ ]:
# 2. 产品embedding（产品描述->向量+cosine相似度检索）
product_texts = [p["desc"] for p in products]
product_embeddings = model.encode(product_texts, show_progress_bar=False)

query = "提亮肤色的精华液"
query_emb = model.encode([query])
similarities = cosine_similarity(query_emb, product_embeddings)[0]
top_product_idx = int(np.argmax(similarities))

print(f"产品embedding维度: {product_embeddings.shape}")
print(f"查询: {query}")
print(f"最相似产品: {products[top_product_idx]['name']} (相似度={similarities[top_product_idx]:.4f})")
print("\n全部产品相似度排序:")
for idx in np.argsort(similarities)[::-1]:
    print(f"  {products[idx]['name']}: {similarities[idx]:.4f}")

In [ ]:
# 3. 内容embedding（营销文案->向量+相似文案推荐）
copy_embeddings = model.encode(marketing_copy, show_progress_bar=False)
copy_sim_matrix = cosine_similarity(copy_embeddings)

print(f"文案embedding维度: {copy_embeddings.shape}")
print("\n相似文案推荐:")
for i, copy in enumerate(marketing_copy):
    sim_scores = copy_sim_matrix[i].copy()
    sim_scores[i] = -1  # 排除自己
    most_similar_idx = int(np.argmax(sim_scores))
    print(f"  文案{i+1} -> 文案{most_similar_idx+1} (相似度={sim_scores[most_similar_idx]:.4f})")
    print(f"    原文: {copy[:40]}...")
    print(f"    推荐: {marketing_copy[most_similar_idx][:40]}...")

## 3. Two-Tower 跨域对齐模型

**问题**：客户 embedding 和产品 embedding 是分别编码的，它们的向量空间不"对齐"--直接计算 cosine 相似度没有意义。

**Two-Tower 解决方案**：用两个 MLP 塔（Tower A for 客户, Tower B for 产品）将各自的 embedding 映射到**共享的64维空间**，然后用 InfoNCE 对比损失训练，使正样本（客户实际购买的产品）的相似度高、负样本的相似度低。

```
客户向量(384维) -> Tower A (MLP) -> 客户向量(64维) ─┐
                                                    ├──-> cos(u,v) -> InfoNCE
产品向量(384维) -> Tower B (MLP) -> 产品向量(64维) ─┘
```

**InfoNCE 损失**（对比学习核心）：
```
L = -log[ exp(sim(u, v⁺)) / Σ exp(sim(u, vᵢ)) ]
```
- v⁺ 是正样本（对角线），vᵢ 包括正样本和所有负样本（batch内其他产品）
- 本质是一个 B 分类问题：在 B 个候选产品中识别出正样本
- 训练后，客户向量和产品向量在同一空间可直接比较--这就是推荐系统的核心

## 4：Two-Tower双塔模型（torch实现，客户-产品匹配检索+InfoNCE）

In [ ]:
# 4. Two-Tower双塔模型（torch实现，客户-产品匹配检索+InfoNCE）
class TwoTowerModel(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=128, embed_dim=64):
        super().__init__()
        self.user_tower = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.item_tower = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, user_feat, item_feat):
        user_emb = self.user_tower(user_feat)
        item_emb = self.item_tower(item_feat)
        # L2归一化，使点积=余弦相似度
        user_emb = F.normalize(user_emb, p=2, dim=1)
        item_emb = F.normalize(item_emb, p=2, dim=1)
        return user_emb, item_emb

def info_nce_loss(user_emb, item_emb, temperature=0.1):
    # logits[i][j] = 客户i与产品j的相似度 / temperature
    logits = torch.matmul(user_emb, item_emb.t()) / temperature  # [B, B]
    # 正样本在对角线：客户i 对应 产品i
    labels = torch.arange(len(user_emb))
    loss = F.cross_entropy(logits, labels)
    return loss

# 训练准备
tt_model = TwoTowerModel(input_dim=384)
optimizer = torch.optim.Adam(tt_model.parameters(), lr=0.01)
user_tensor = torch.tensor(customer_embeddings, dtype=torch.float32)
item_tensor = torch.tensor(product_embeddings, dtype=torch.float32)

# 训练100步
for epoch in range(100):
    optimizer.zero_grad()
    user_emb, item_emb = tt_model(user_tensor, item_tensor)
    loss = info_nce_loss(user_emb, item_emb, temperature=0.1)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 检索：客户C001 -> 最相似产品
tt_model.eval()
with torch.no_grad():
    user_emb, item_emb = tt_model(user_tensor, item_tensor)
    sim = torch.matmul(user_emb, item_emb.T)
    c001_top = int(sim[0].argmax())
    print(f"\n客户C001 -> 最相似产品: {products[c001_top]['name']} (相似度={sim[0][c001_top]:.4f})")
    print(f"  客户行为: {customers[0]['text'][:40]}...")
    print(f"  产品描述: {products[c001_top]['desc'][:40]}...")

## 4. CLIP 图文对齐与多模态演进

### CLIP：对比学习对齐图文

CLIP（Contrastive Language-Image Pre-training）是 OpenAI 2021 年发布的模型，用对比学习将图像和文本对齐到同一向量空间。它的架构和 Two-Tower 本质相同--两个编码器（图像+文本）+ 对比损失。

```
产品图片 -> 图像编码器(ViT) -> 图像向量 ─┐
                                        ├──-> cos相似度 -> 对比损失
产品描述 -> 文本编码器(Transformer) -> 文本向量 ─┘
```

**营销应用**：给一张产品图片和几段描述，CLIP 能判断哪段描述最匹配--这是多模态搜索的基础。

### 从 CLIP 到 GPT-4o 的演进

| 阶段 | 代表方法 | 核心思想 |
|:----:|---------|---------|
| 对比学习对齐 | CLIP (2021) | 双塔，对比损失对齐图文 |
| 视觉-语言预训练 | BLIP-2 (2023) | Q-Former桥接视觉编码器和LLM |
| 原生多模态 | GPT-4o (2024) | 端到端统一token空间 |
| 开源多模态 | LLaVA (2024) | CLIP-ViT+投影层+LLM |

## 5-6：CLIP图文对齐 + 多模态演进分析表

In [ ]:
# 5. CLIP图文对齐（transformers CLIPModel，产品图-文相似度）
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# 用PIL生成3张产品图片（纯色块代表产品照片）
clip_images = [
    Image.new('RGB', (224, 224), color=(200, 0, 0)),    # 红色 - 口红
    Image.new('RGB', (224, 224), color=(0, 100, 200)),   # 蓝色 - 防晒霜
    Image.new('RGB', (224, 224), color=(255, 200, 0)),   # 黄色 - 精华液
]
clip_texts = [
    "一支红色丝绒口红",
    "一瓶蓝色防晒霜",
    "一瓶黄色精华液",
]

# 提取图文特征
inputs = clip_processor(text=clip_texts, images=clip_images, return_tensors="pt", padding=True)
with torch.no_grad():
    text_features = clip_model.get_text_features(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]
    )
    image_features = clip_model.get_image_features(pixel_values=inputs["pixel_values"])

# 归一化后计算相似度矩阵
text_features = text_features / text_features.norm(dim=-1, keepdim=True)
image_features = image_features / image_features.norm(dim=-1, keepdim=True)
clip_sim = torch.matmul(image_features, text_features.T)

print("CLIP图文相似度矩阵:")
for i in range(len(clip_images)):
    best_text = int(clip_sim[i].argmax())
    print(f"  图片{i+1}({['红','蓝','黄'][i]}) -> {clip_texts[best_text]} (相似度={clip_sim[i][best_text]:.4f})")
print("\n解读：CLIP用对比学习将图文对齐到同一空间，能判断哪段文字最匹配哪张图片。")
print("注意：PIL纯色图片缺乏纹理，CLIP匹配主要依赖颜色-文本关联，效果可能不如真实产品照片。")

In [ ]:
# 6. 多模态演进分析表（CLIP->BLIP-2->GPT-4o->LLaVA，对比架构/能力/营销应用）
evolution_data = [
    {
        "阶段": "对比学习对齐 (2021)",
        "代表方法": "CLIP",
        "架构": "双塔：图像编码器(ViT)+文本编码器(Transformer)，对比损失对齐",
        "核心能力": "图文匹配、零样本分类、多模态检索",
        "局限": "只能匹配不能生成；细粒度视觉理解弱",
        "营销应用": "商品图文匹配、多模态搜索、广告素材自动标注",
    },
    {
        "阶段": "视觉-语言预训练 (2023)",
        "代表方法": "BLIP-2",
        "架构": "Q-Former桥接冻结视觉编码器+冻结LLM",
        "核心能力": "图文理解+文案生成、图像问答",
        "局限": "Q-Former是信息瓶颈；视觉细节丢失",
        "营销应用": "看图写文案、产品图描述生成、图片问答客服",
    },
    {
        "阶段": "原生多模态 (2024-2025)",
        "代表方法": "GPT-4o / Gemini",
        "架构": "端到端统一token空间，无编码后对齐步骤",
        "核心能力": "跨模态细微关联理解、图文音视频统一处理",
        "局限": "闭源API、成本高、数据隐私顾虑",
        "营销应用": "自动广告创意生成、视频内容理解、多模态客服",
    },
    {
        "阶段": "开源多模态 (2024)",
        "代表方法": "LLaVA",
        "架构": "CLIP-ViT视觉编码器+线性投影层+LLaMA LLM",
        "核心能力": "开源可私有部署、图文理解+生成",
        "局限": "能力不及GPT-4o；投影层简单",
        "营销应用": "低成本多模态方案、数据敏感场景的图文分析",
    },
]

evolution_df = pd.DataFrame(evolution_data)
for _, row in evolution_df.iterrows():
    print(f"【{row['阶段']}】{row['代表方法']}")
    print(f"  架构: {row['架构']}")
    print(f"  能力: {row['核心能力']}")
    print(f"  局限: {row['局限']}")
    print(f"  营销: {row['营销应用']}")
    print()

## 5. 反思与前沿

### 反思问题
1. 你的营销场景中，四大表示类型哪个最难构建？瓶颈在数据、模型还是融合策略？
2. Two-Tower 模型训练后，客户C001的最相似产品是否是P001（正样本）？如果不是，可能是什么原因？（提示：训练步数不足/负采样不充分/embedding维度过低）
3. CLIP 的图文匹配结果是否完美？哪些因素会导致误匹配？（提示：PIL纯色图片缺乏纹理/CLIP训练数据偏自然照片/颜色-文本关联的文化差异）
4. 从 CLIP 到 GPT-4o，"原生多模态"相比"双塔对齐"的本质优势是什么？什么场景下 CLIP 的双塔架构反而更优？（提示：检索效率/预计算/在线服务）

### 2026 前沿：CLIP对比学习 -> GPT-4o原生多模态 -> LLaVA开源
- **CLIP**（2021）：对比学习双塔，训练高效、检索方便，但只能匹配不能生成
- **BLIP-2**（2023）：Q-Former桥接冻结视觉编码器和LLM，低成本实现图文理解+生成
- **GPT-4o**（2024-2025）：原生多模态，统一token空间处理图文音频，能理解跨模态细微关联
- **LLaVA**（2024）：开源方案，CLIP-ViT+投影层+LLM，可私有部署适合数据敏感场景

**对比学习是贯穿全程的底层技术**：CLIP用对比学习对齐图文，Two-Tower用对比学习对齐客户-产品，sentence-transformers用对比学习微调文本编码器。

参考 [CLIP论文](https://arxiv.org/abs/2103.00020) + [BLIP-2论文](https://arxiv.org/abs/2301.12597) + [LLaVA论文](https://arxiv.org/abs/2304.08485) + [sentence-transformers](https://www.sbert.net/)。